# 02 — Zero-shot and few-shot baselines

**Purpose:** Evaluate the unchanged Qwen model in Urdu and English; few-shot demonstrations come ONLY from a training split.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/Urdu_Qwen_Research_Notebooks'), Path('/content/Urdu_Qwen_Research_Notebooks')]
ROOT = next((p for p in CANDIDATES if (p / 'project_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Unzip the entire project folder into /kaggle/working/ or /content/, then run this notebook. See README.md.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print('Project folder:', ROOT)

Run `01_preprocess_data` first. Zero-shot means zero labeled examples in the prompt. Few-shot uses selected **training** examples as demonstrations; these are not gradient updates. Keep greedy decoding and the same exact allowed labels throughout all experiments.

In [ ]:
from collections import Counter
BASE_MODEL = cfg()['model_id']
EVAL_SPLIT = 'validation'  # use test only after deciding hyperparameters
LANG = 'ur'
N_FEWSHOT_PER_CLASS = 1
MAX_ROWS = None  # use 20 for a smoke run, None for complete split
labels = task_labels('sib200')
rows = load_task('sib200', LANG, EVAL_SPLIT)
if MAX_ROWS is not None: rows = rows[:MAX_ROWS]
print('Rows:',len(rows),'Labels:',labels)
model, tokenizer = load_model(BASE_MODEL)
print('Model loaded.')

In [ ]:
demonstrations = examples_for('sib200', N_FEWSHOT_PER_CLASS, 42, lang=LANG)
assert all(r['split']=='train' for r in demonstrations)
print('Demonstrations from training only:',Counter(r['label'] for r in demonstrations))
for condition, demos in [('zero_shot',None),('few_shot',demonstrations)]:
    output = infer_rows(model, tokenizer, rows, labels, LANG, cfg()['max_new_tokens'], demos)
    path = ROOT/'outputs/predictions'/f'sib200_{condition}_{LANG}_{EVAL_SPLIT}.csv'
    meta = dict(task='sib200',lang=LANG,split=EVAL_SPLIT,run_id=condition,model_id=BASE_MODEL,
                dataset_sha256=dataset_sha('sib200',LANG,EVAL_SPLIT),
                subset=(MAX_ROWS is not None),n_fewshot_per_class=N_FEWSHOT_PER_CLASS if demos else 0)
    save_prediction_run(path,output,meta)
    print(condition, len(output), 'saved to',path, 'valid',sum(x['valid'] for x in output))
    print('First five:',[(r['reference'],r['prediction'],r['raw_prediction']) for r in output[:5]])

When done, release GPU memory before training. **Do not use validation performance as published final test performance.**

In [ ]:
del model
import torch
if torch.cuda.is_available(): torch.cuda.empty_cache()